Version 0: Circuit that constructs the superposition: $\sum_{2^I}(|Window_i>|Index_i>|Grid>)$
- 1-dimensional network.

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
print(qiskit.__version__)

In [ ]:
N = 4
M = 2
W = N - M + 1
k = int(np.ceil(np.log2(W))) if W > 1 else 1
print(f"Posibles soluciones W: {W}, qubits necesarios para representar W posiciones -> k: {k}")

In [ ]:
# n: posiciones libres/ocupadas
n = QuantumRegister(N, "n")
# m: trabajo (tamano M)
m = QuantumRegister(M, "m")
# idx: seleccion de ventana (tamano k)
idx = QuantumRegister(k, "i")

qc = QuantumCircuit(n, idx, m)

# ---------------------------
# Estado inicial de ejemplo para n
qc.x(n[2])
qc.x(n[3])
# qc.x(n[7])
# qc.x(n[13])
# ---------------------------

# Superposicion uniforme de indices
qc.h(idx)

# Para cada ventana valida i = 0..W-1:
# 1) aplicar X en idx[b] cuando el bit b de i sea 0 (control sobre |idx> == |i>)
# 2) copiar (XOR) la ventana n[i:i+M] sobre m[0:M]
# 3) deshacer las X
for i in range(W):
    bits = [(i >> b) & 1 for b in range(k)]  # idx[0] es LSB, idx[k-1] es MSB -> little-endian

    # Activar control de igualdad idx == i
    for b in range(k):
        if bits[b] == 0:
            qc.x(idx[b])

    # XOR de la ventana seleccionada sobre m
    for j in range(M):
        controls = [idx[b] for b in range(k)] + [n[i + j]]
        qc.mcx(controls, m[j])

    # Deshacer X
    for b in range(k):
        if bits[b] == 0:
            qc.x(idx[b])

qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# Quita los qubits n (asumimos que son los N primeros: q0..q{N-1})
rho = partial_trace(sv, list(range(N)))

# Comprobar pureza: Tr(rho^2)=1 <=> puro
purity = np.real(np.trace(rho.data @ rho.data))
print(f"Purity = {purity}")

if not np.isclose(purity, 1.0, atol=1e-10):
    raise ValueError("El estado reducido no es puro; no se puede representar como un unico ket.")

# Obtener array de posiciones (qubits n) desde el estado completo
# En la cadena binaria de Qiskit: ... n_{N-1}...n_0 (al final)
# Mostramos como n_0...n_{N-1} para que coincida con n[0], n[1], ...
tol_n = 1e-10
total_q = qc.num_qubits
array_posiciones = None

for basis_idx, amp in enumerate(sv.data):
    if abs(amp) <= tol_n:
        continue

    bits_full = format(basis_idx, f'0{total_q}b')
    n_desc = bits_full[-N:]
    n_asc = n_desc[::-1]

    if array_posiciones is None:
        array_posiciones = n_asc
    elif n_asc != array_posiciones:
        raise ValueError("Los qubits n no son deterministas (no hay un unico array de posiciones).")

if array_posiciones is None:
    raise ValueError("No se pudo inferir el array de posiciones desde el statevector.")

print(f"Array de posiciones = {array_posiciones}\n")

# Extraer el ket (autovector del autovalor ~1)
vals, vecs = np.linalg.eigh(rho.data)
psi_red = Statevector(vecs[:, np.argmax(vals)])

# Mostrar SOLO etiquetas (sin amplitudes), ordenadas por indices
# Convencion de bits en psi_red: m_{M-1}...m_0 i_{k-1}...i_0
total_bits = M + k
tol = 1e-10
filas = []

for basis_idx, amp in enumerate(psi_red.data):
    if abs(amp) <= tol:
        continue

    bits = format(basis_idx, f'0{total_bits}b')
    m_desc = bits[:M]           # m_{M-1}...m_0
    i_desc = bits[M:]           # i_{k-1}...i_0

    ventana = m_desc[::-1]      # m_0...m_{M-1}
    indices = i_desc            # orden natural por indices

    filas.append((int(indices, 2), indices, ventana))

filas.sort(key=lambda x: (x[0], x[2]))

# Primeros W resultados: ventanas validas
for _, indices, ventana in filas[:W]:
    print(f"Indices: {indices} || Ventana: {ventana}\n")

# Resto de resultados (fuera del rango valido de ventanas)
if len(filas) > W:
    print("----------------------------------------")
    print("Resto (indices fuera de ventana valida):\n")
    for _, indices, ventana in filas[W:]:
        print(f"Indices: {indices} || Ventana: {ventana}\n")